In [ ]:
import requests
import pandas as pd
import json
from datetime import datetime

API_KEY = your api key

# WCOnline requires YYYYMMDD format
today = datetime.now().strftime("%Y%m%d")

request_type = "CUSTOM"         # AVAIL, SCHED, APPTS, STAFF, CUSTOM
request_date = 20250911           # auto-today

url = f"https://gannon.mywconline.com/api?type={request_type}&date={request_date}"

headers = {"Authorization": f"Bearer {API_KEY}"}

response = requests.get(url, headers=headers)
response_json = response.json()

print("RAW RESPONSE:")
print(json.dumps(response_json, indent=4))

df = None  # placeholder

# Convert if it's a list
if isinstance(response_json, list):
    df = pd.DataFrame(response_json)
    print("\nFull DataFrame:")
    display(df)

# If response is a dict with a list inside
elif isinstance(response_json, dict):
    for key, value in response_json.items():
        if isinstance(value, list):
            print(f"\nDetected list under key '{key}':")
            df = pd.DataFrame(value)
            display(df)
            break
    if df is None:
        print("No list-style data returned:", response_json)

else:
    print("Unexpected format:", type(response_json))


# ---------------------------------------------------
# ⭐ FILTER FOR STEM CENTER (case-insensitive)
# ---------------------------------------------------
if df is not None and "Schedule Title" in df.columns:
    stem_df = df[df["Schedule Title"].str.upper() == "STEM CENTER"]
    
    print("\n\n⭐ STEM CENTER FILTERED RESULTS ⭐")
    display(stem_df)
else:
    print("\n⚠️ Cannot filter — DataFrame empty or missing 'Schedule Title' column.")
